In [6]:
# Drive Mounting Check
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
#  STEP 1: restore local copy, normalise names, verify against manifest
from pathlib import Path
import shutil, pandas as pd

PROJECT   = Path("/content/drive/MyDrive/0_potato_project_v1")
CACHE     = PROJECT / "data" / "potato_raw" / "raw"
DATA_ROOT = Path("/content/potato")
MANIFEST  = PROJECT / "results" / "audit" / "manifest.csv"

df = pd.read_csv(MANIFEST)
labels = sorted(df.label.unique())
print("manifest labels:", labels)

if DATA_ROOT.exists() and any(DATA_ROOT.iterdir()):
    print("local copy present")
else:
    print("restoring from Drive...")
    shutil.copytree(CACHE, DATA_ROOT)
    print("restored")

have = {d.name.lower(): d for d in DATA_ROOT.iterdir() if d.is_dir()}
for lab in labels:
    if (DATA_ROOT / lab).exists():
        continue
    key = lab.split("___")[-1].lower()
    src = have.get(key)
    if src is None:
        print(f"  !! no folder on disk matching '{lab}'")
    else:
        src.rename(DATA_ROOT / lab)
        print(f"  {src.name} -> {lab}")

print("\nfolders now:", sorted(d.name for d in DATA_ROOT.iterdir() if d.is_dir()))

print("\nshape:", df.shape)
print(df[["label", "keep", "group_id", "fold"]].dtypes.to_string())
print("\nimages per class:")
print(df.label.value_counts().to_string())
print("\nfold x class:")
print(pd.crosstab(df.fold, df.label).to_string())
print(f"\nkeep=True: {df.keep.sum()}   groups: {df.group_id.nunique()}")

missing = sum(1 for p in df.path if not (DATA_ROOT / p).exists())
print(f"\nmissing files (all {len(df)} checked): {missing}")

manifest labels: ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']
restoring from Drive...
restored
  Early_blight -> Potato___Early_blight
  Late_blight -> Potato___Late_blight
  Healthy -> Potato___healthy

folders now: ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']

shape: (2152, 12)
label       object
keep          bool
group_id     int64
fold         int64

images per class:
label
Potato___Early_blight    1000
Potato___Late_blight     1000
Potato___healthy          152

fold x class:
label  Potato___Early_blight  Potato___Late_blight  Potato___healthy
fold                                                                
0                        200                   199                30
1                        201                   200                31
2                        200                   200                31
3                        200                   200                30
4                        199                   20

In [8]:

# STEP 2: Class Mapping
# 0 -> Potato___Early_blight, 1 -> Potato___Late_blight, 2 -> Potato___healthy
import json

CLASSES = sorted(df.label.unique())              # deterministic, not insertion order
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

df["y"] = df.label.map(CLASS_TO_IDX)

CLASSES_JSON = PROJECT / "results" / "classes.json"
CLASSES_JSON.write_text(json.dumps(
    {"classes": CLASSES, "class_to_idx": CLASS_TO_IDX}, indent=2))

for c, i in CLASS_TO_IDX.items():
    print(f"  {i} -> {c}")
print(f"\nNUM_CLASSES = {NUM_CLASSES}")
print("y dtype:", df.y.dtype, " nulls:", int(df.y.isna().sum()))
print("\ncounts by y:")
print(df.y.value_counts().sort_index().to_string())
print(f"\nsaved -> {CLASSES_JSON}")

  0 -> Potato___Early_blight
  1 -> Potato___Late_blight
  2 -> Potato___healthy

NUM_CLASSES = 3
y dtype: int64  nulls: 0

counts by y:
y
0    1000
1    1000
2     152

saved -> /content/drive/MyDrive/0_potato_project_v1/results/classes.json


In [ ]:
# 03 STEP 3: fold -> (train, inner-val, eval) row indices
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

SEED         = 42
N_FOLDS      = 5
INNER_SPLITS = 7

def get_indices(df, fold, seed=SEED):
    """Row positions into df. eval = held-out fold, never seen during training."""
    d = df[df.keep].reset_index(drop=True)

    ev = np.flatnonzero(d.fold.values == fold)
    pool = np.flatnonzero(d.fold.values != fold)

    sgkf = StratifiedGroupKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(sgkf.split(
        pool, d.y.values[pool], groups=d.group_id.values[pool]))

    return pool[tr_rel], pool[va_rel], ev

# verify every fold
d = df[df.keep].reset_index(drop=True)
for k in range(N_FOLDS):
    tr, va, ev = get_indices(df, k)
    g = lambda i: set(d.group_id.values[i])
    assert not (g(tr) & g(va)) and not (g(tr) & g(ev)) and not (g(va) & g(ev))
    assert len(tr) + len(va) + len(ev) == len(d)
    c = lambda i: np.bincount(d.y.values[i], minlength=3)
    print(f"fold {k}:  train {len(tr):>4} {c(tr)}   val {len(va):>3} {c(va)}   eval {len(ev):>3} {c(ev)}")

fold 0:  train 1477 [687 687 103]   val 246 [113 114  19]   eval 429 [200 199  30]
fold 1:  train 1474 [684 686 104]   val 246 [115 114  17]   eval 432 [201 200  31]
fold 2:  train 1476 [686 686 104]   val 245 [114 114  17]   eval 431 [200 200  31]
fold 3:  train 1477 [687 685 105]   val 245 [113 115  17]   eval 430 [200 200  30]
fold 4:  train 1477 [687 685 105]   val 245 [114 114  17]   eval 430 [199 201  30]


In [10]:
# STEP 4: transforms  (AUG_MODE: 'baseline' | 'aggressive' | 'greyworld')
import torch
from torchvision import transforms as T

IMG_SIZE  = 224
IMNET_MEAN, IMNET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
AUG_MODE  = "baseline"

class GreyWorld:
    """Scale each channel so its mean matches the image's overall mean."""
    def __call__(self, x):                      # x: float tensor [3,H,W] in [0,1]
        m = x.mean(dim=(1, 2), keepdim=True)
        return (x * m.mean() / m.clamp(min=1e-6)).clamp(0, 1)

def build_transforms(mode=AUG_MODE):
    gw = [GreyWorld()] if mode == "greyworld" else []

    geo = ([T.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0), ratio=(0.85, 1.18))]
           if mode == "aggressive" else
           [T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.11))])

    train = T.Compose(geo + [
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
        T.RandomRotation(20),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),
        T.ToTensor(), *gw, T.Normalize(IMNET_MEAN, IMNET_STD),
    ])
    eval_ = T.Compose([
        T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE),
        T.ToTensor(), *gw, T.Normalize(IMNET_MEAN, IMNET_STD),
    ])
    return train, eval_

# verify on one real image
from PIL import Image
tt, te = build_transforms()
img = Image.open(DATA_ROOT / df.path.iloc[0]).convert("RGB")
a, b = tt(img), te(img)
print("mode:", AUG_MODE)
print(f"train tensor {tuple(a.shape)} {a.dtype}  range [{a.min():.2f}, {a.max():.2f}]")
print(f"eval  tensor {tuple(b.shape)} {b.dtype}  range [{b.min():.2f}, {b.max():.2f}]")
print("two train draws identical:", torch.equal(tt(img), tt(img)))
print("two eval  draws identical:", torch.equal(te(img), te(img)))

mode: baseline
train tensor (3, 224, 224) torch.float32  range [-2.12, 2.43]
eval  tensor (3, 224, 224) torch.float32  range [-2.08, 1.92]
two train draws identical: False
two eval  draws identical: True


In [12]:
# STEP 5: Dataset
from torch.utils.data import Dataset

class PotatoDataset(Dataset):
    def __init__(self, frame, indices, root, transform):
        self.paths = frame.path.values[indices]
        self.ys    = frame.y.values[indices].astype("int64")
        self.root, self.tf = root, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.root / self.paths[i]) as im:
            x = self.tf(im.convert("RGB"))
        return x, int(self.ys[i])

# verify
tr, va, ev = get_indices(df, 0)
tf_train, tf_eval = build_transforms()

ds_tr = PotatoDataset(d, tr, DATA_ROOT, tf_train)
ds_ev = PotatoDataset(d, ev, DATA_ROOT, tf_eval)

x, y = ds_tr[0]
print(f"len train {len(ds_tr)}  eval {len(ds_ev)}")
print(f"item: {tuple(x.shape)} {x.dtype}, label {y} ({CLASSES[y]})")
print("label dtype ok:", isinstance(y, int))

# every label in range, and the split's class counts still match step 3
print("train counts:", np.bincount(ds_tr.ys, minlength=3))
print("eval  counts:", np.bincount(ds_ev.ys, minlength=3))

len train 1477  eval 429
item: (3, 224, 224) torch.float32, label 0 (Potato___Early_blight)
label dtype ok: True
train counts: [687 687 103]
eval  counts: [200 199  30]


In [13]:
# STEP 6: class weights (chosen) + sampler (available, unused)
from torch.utils.data import WeightedRandomSampler

BALANCE = "loss"

def class_weights(ys, n=NUM_CLASSES):
    counts = np.bincount(ys, minlength=n).astype("float64")
    w = counts.sum() / (n * counts)        # sklearn 'balanced' convention
    return torch.tensor(w, dtype=torch.float32)

def make_sampler(ys, n=NUM_CLASSES):
    counts = np.bincount(ys, minlength=n)
    per_item = (1.0 / counts)[ys]
    return WeightedRandomSampler(torch.as_tensor(per_item, dtype=torch.double),
                                 num_samples=len(ys), replacement=True)

# verify on fold 0
w = class_weights(ds_tr.ys)
print("counts :", np.bincount(ds_tr.ys, minlength=3))
print("weights:", [round(v, 3) for v in w.tolist()])
print("healthy penalty vs early blight:", round((w[2] / w[0]).item(), 2), "x")
print("\nBALANCE =", BALANCE)

counts : [687 687 103]
weights: [0.717, 0.717, 4.78]
healthy penalty vs early blight: 6.67 x

BALANCE = loss


In [14]:
# STEP 8: the one function 04_train calls
from torch.utils.data import DataLoader

BATCH_SIZE, NUM_WORKERS = 32, 2

def _seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); __import__("random").seed(s)

def get_loaders(frame, fold, aug_mode=AUG_MODE, batch_size=BATCH_SIZE, seed=SEED):
    d = frame[frame.keep].reset_index(drop=True)
    tr, va, ev = get_indices(frame, fold, seed)
    tf_tr, tf_ev = build_transforms(aug_mode)

    g = torch.Generator(); g.manual_seed(seed + fold)
    common = dict(num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                  worker_init_fn=_seed_worker)

    train = DataLoader(PotatoDataset(d, tr, DATA_ROOT, tf_tr), batch_size=batch_size,
                       shuffle=True, drop_last=True, generator=g, **common)
    val   = DataLoader(PotatoDataset(d, va, DATA_ROOT, tf_ev), batch_size=batch_size,
                       shuffle=False, **common)
    test  = DataLoader(PotatoDataset(d, ev, DATA_ROOT, tf_ev), batch_size=batch_size,
                       shuffle=False, **common)

    return train, val, test, class_weights(d.y.values[tr])

# verify
tl, vl, el, w = get_loaders(df, 0)
xb, yb = next(iter(tl))
print(f"batches  train {len(tl)}  val {len(vl)}  eval {len(el)}")
print(f"batch    {tuple(xb.shape)} {xb.dtype}   labels {tuple(yb.shape)} {yb.dtype}")
print(f"range    [{xb.min():.2f}, {xb.max():.2f}]   labels seen {sorted(set(yb.tolist()))}")
print(f"weights  {[round(v,3) for v in w.tolist()]}")
print(f"device   {'cuda' if torch.cuda.is_available() else 'cpu'}")

batches  train 46  val 8  eval 14
batch    (32, 3, 224, 224) torch.float32   labels (32,) torch.int64
range    [-2.12, 2.64]   labels seen [0, 1, 2]
weights  [0.717, 0.717, 4.78]
device   cpu


In [16]:
# STEP 8: leakage gates
def check_fold(frame, fold, seed=SEED):
    d = frame[frame.keep].reset_index(drop=True)
    tr, va, ev = get_indices(frame, fold, seed)
    G, P = d.group_id.values, d.path.values

    for a, b, name in [(tr, va, "train/val"), (tr, ev, "train/eval"), (va, ev, "val/eval")]:
        assert not (set(G[a]) & set(G[b])), f"fold {fold}: group leak in {name}"
        assert not (set(P[a]) & set(P[b])), f"fold {fold}: path leak in {name}"

    assert len(tr) + len(va) + len(ev) == len(d), f"fold {fold}: rows lost"
    assert np.bincount(d.y.values[ev], minlength=3).min() > 0, f"fold {fold}: empty eval class"
    return len(tr), len(va), len(ev)

for k in range(N_FOLDS):
    print(f"fold {k}: {check_fold(df, k)}  ok")

# every image is evaluated exactly once across the five folds
seen = np.concatenate([get_indices(df, k)[2] for k in range(N_FOLDS)])
assert len(seen) == len(np.unique(seen)) == len(d), "CV coverage broken"
print(f"\nCV coverage: {len(seen)} images, each evaluated exactly once")

fold 0: (1477, 246, 429)  ok
fold 1: (1474, 246, 432)  ok
fold 2: (1476, 245, 431)  ok
fold 3: (1477, 245, 430)  ok
fold 4: (1477, 245, 430)  ok

CV coverage: 2152 images, each evaluated exactly once
